# Baseline: TF-IDF + Logistic Regression — SemEval-2014 Laptop ABSA

Task: aspect-term polarity classification (given a sentence and a known aspect term, predict `positive`/`negative`/`neutral`/`conflict`).

- Dataset: SemEval-2014 Task 4, Laptop domain, `Laptop_Train_v2.xml` (3045 sentences, official train split).
- No official gold test file is available (see `docs/Dataset-Verification-Report.md` in the repo) — this notebook holds out a 15% dev split from the train file instead, with a fixed seed for reproducibility.
- On Kaggle: add the dataset `charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis` as a notebook input — it contains `Laptop_Train_v2.xml` (verified byte-identical to the official alt.qcri.org download).


In [ ]:
import glob
import json
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split


## 1. Locate the data

Works both on Kaggle (`/kaggle/input/...`) and locally (repo `data/raw/train/`).

In [ ]:
candidates = glob.glob("/kaggle/input/**/Laptop_Train_v2.xml", recursive=True) + \
             glob.glob("../data/raw/train/Laptop_Train_v2.xml") + \
             glob.glob("data/raw/train/Laptop_Train_v2.xml")

if not candidates:
    raise FileNotFoundError(
        "Laptop_Train_v2.xml not found. On Kaggle: add the "
        "'charitarth/semeval-2014-task-4-aspectbasedsentimentanalysis' dataset as input. "
        "Locally: place the file under data/raw/train/."
    )

TRAIN_XML_PATH = candidates[0]
print("Using:", TRAIN_XML_PATH)


## 2. Parse the SemEval XML

Same logic as `src/data/semeval_loader.py` in the repo, inlined here so the notebook is self-contained.

In [ ]:
@dataclass
class AspectTerm:
    term: str
    polarity: str
    start: int
    end: int


@dataclass
class Sentence:
    sentence_id: str
    text: str
    aspect_terms: list = field(default_factory=list)


def load_semeval_xml(path):
    """Load a SemEval-2014 Task 4 XML file (e.g. Laptop_Train_v2.xml).

    Aspect terms without a `polarity` attribute (unlabeled blind test data)
    are skipped; the sentence itself is still returned.
    """
    root = ET.parse(path).getroot()
    sentences = []
    for sent_el in root.findall("sentence"):
        text = sent_el.findtext("text") or ""
        aspect_terms = []
        for term_el in sent_el.findall("./aspectTerms/aspectTerm"):
            polarity = term_el.get("polarity")
            if polarity is None:
                continue
            aspect_terms.append(
                AspectTerm(
                    term=term_el.get("term", ""),
                    polarity=polarity,
                    start=int(term_el.get("from", -1)),
                    end=int(term_el.get("to", -1)),
                )
            )
        sentences.append(
            Sentence(sentence_id=sent_el.get("id", ""), text=text, aspect_terms=aspect_terms)
        )
    return sentences


sentences = load_semeval_xml(TRAIN_XML_PATH)
print(f"Loaded {len(sentences)} sentences")
print(f"Total aspect terms: {sum(len(s.aspect_terms) for s in sentences)}")


## 3. Preprocess: flatten into (context, aspect, polarity) examples

Same logic as `src/data/preprocess.py`.

In [ ]:
VALID_POLARITIES = {"positive", "negative", "neutral", "conflict"}


def mark_aspect(text, start, end):
    """Wrap the aspect span in $T$ markers so the vectorizer can pick up on it."""
    if start < 0 or end < 0 or end > len(text) or start >= end:
        return text
    return f"{text[:start]}$T$ {text[start:end]} $T${text[end:]}"


def build_examples(sentence_list):
    contexts, aspects, labels = [], [], []
    for sent in sentence_list:
        for term in sent.aspect_terms:
            if term.polarity not in VALID_POLARITIES:
                continue
            contexts.append(mark_aspect(sent.text, term.start, term.end))
            aspects.append(term.term)
            labels.append(term.polarity)
    return contexts, aspects, labels


## 4. Train / dev split

No official gold test available yet, so we hold out 15% of sentences (fixed seed) as a dev proxy. Swap in a real gold XML later by re-running `load_semeval_xml` on it instead of splitting.

In [ ]:
DEV_RATIO = 0.15
SEED = 42

train_sentences, dev_sentences = train_test_split(sentences, test_size=DEV_RATIO, random_state=SEED)

train_ctx, train_asp, train_labels = build_examples(train_sentences)
dev_ctx, dev_asp, dev_labels = build_examples(dev_sentences)

print(f"Train examples: {len(train_labels)} | Dev examples: {len(dev_labels)}")


## 5. Baseline model: TF-IDF(sentence with `$T$`-marked aspect) + TF-IDF(aspect term) → Logistic Regression

In [ ]:
class TfidfLogRegBaseline:
    def __init__(self, max_features=20000, ngram_range=(1, 2)):
        self.context_vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
        self.aspect_vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2))
        self.clf = LogisticRegression(max_iter=1000, class_weight="balanced")

    def _vectorize(self, contexts, aspects, fit):
        if fit:
            x_ctx = self.context_vectorizer.fit_transform(contexts)
            x_asp = self.aspect_vectorizer.fit_transform(aspects)
        else:
            x_ctx = self.context_vectorizer.transform(contexts)
            x_asp = self.aspect_vectorizer.transform(aspects)
        return hstack([x_ctx, x_asp])

    def fit(self, contexts, aspects, labels):
        x = self._vectorize(contexts, aspects, fit=True)
        self.clf.fit(x, labels)
        return self

    def predict(self, contexts, aspects):
        x = self._vectorize(contexts, aspects, fit=False)
        return self.clf.predict(x)


model = TfidfLogRegBaseline().fit(train_ctx, train_asp, train_labels)
preds = model.predict(dev_ctx, dev_asp)


## 6. Evaluate

In [ ]:
accuracy = float(np.mean(preds == np.array(dev_labels)))
macro_f1 = f1_score(dev_labels, preds, average="macro", zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro-F1: {macro_f1:.4f}")
print()
print(classification_report(dev_labels, preds, zero_division=0))


## 7. Save metrics

In [ ]:
out_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("results")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "baseline_metrics.json"

out_path.write_text(json.dumps({
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "train_examples": len(train_labels),
    "dev_examples": len(dev_labels),
    "dev_ratio": DEV_RATIO,
    "seed": SEED,
}, indent=2))

print(f"Saved metrics to {out_path}")


## Next steps

- Copy the printed `Accuracy` / `Macro-F1` / classification report back into `plans/project-plan.md` and `docs/Proposal.md` (Metric section) in the main repo.
- If a real official gold test XML is found later, replace the train/dev split in section 4 with `dev_sentences = load_semeval_xml(<gold_test_path>)` and re-run.
